<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0918_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제

스마트폰 사용 데이터를 활용한 스마트폰 중독 여부 예측

- 스마트폰 사용 시간, SNS/게임 이용 시간, 수면 시간, 알림 횟수 등의 행동 데이터를 활용하여 addicted_label을 예측

- XGBoost, Target Encoding을 결합하여 분류 모델 구축

# 데이터

1. Competition 데이터

- train.csv: 691,396행 x 14열

- test.csv: 296,302행 x 13열

- 주요 변수:

age

daily_screen_time_hours

social_media_hours

gaming_hours

work_study_hours

sleep_hours

notifications_per_day

weekend_screen_time

gender, stress_level, academic_work_impact

목표 변수: addiacted_label

2. 외부 참고 데이터

- Smartphone Usage and Addiction 데이터셋 7500행

- 원본 데이터의 분포를 참고하여 추가적인 특징 생성

# 코드 흐름

1. 데이터 로드 & 기본 설정

- Competition 데이터와 원본 참고 덷이터 불러오기

- 수치형 / 범주형 변수 구분

2. 파생변수 생성

- SNS 사용 비율, 게임 사용 비율, 화면시간 / 수면시간 비율 등 생성

- 결측값 개수를 나타내는 missing_count 추가

- 기존 12개 변수 -> 23개 특징

3. Frequency Encoding

- 각 변수의 값이 전체 데이터에서 얼마나 자주 등장하는지 수치화

- Train과 Test의 분포를 함께 사용하되 Target을 사용하지 않아 Target Leackage를 방지

- 특징 수 35개로 증가

4. 원본 데이터 기반 특징 생성

- 원본 데이터의 CDF, 클래스별 CDF 차이, 중앙값과의 거리, Target 평균 등을 추가

- 특징 수 62개로 증가

5. Target Encoding + XGBoost

- 범주형 변수의 실제 값별 Target 정보를 Encoding

- 검증 데이터의 정답이 학습에 들어가지 않도록 Outer Fold + Inner Cross-Fitting 적용

- XGBoost를 5-Fold로 학습하고 여러 Seed를 사용

6. 여러 Seed 결과 결합

- Seed 42와 2026의 예측 결과를 평균

- OOF AUC를 기준으로 최종 방법 선택

- 최종 선택된 probability_average의 OOF AUC는 0.968239

# 주요 코드

1. Target Encoding 설정 - 범주형 데이터를 Target과 관련된 수치로 변환

In [ ]:
encoder=TargetEncoder(
    target_type="binary",
    smooth="auto",
    cv=TE_INNER_FOLDS
)

- TargetEncdor를 이용해 범주형 변수의 값을 Target(스마트폰 중독 여부)과의 관계를 반영한 값으로 변환하는 코드

- cv를 사용하여 내부 교차검증도 적용

2. Cross-Fitting을 이용한 데이터 누수 방지

In [ ]:
encoded_train=encoder.fit_transform(
    X_exact.iloc[train_idx], y[train_idx]
)

encoded_valid=encoder.transform(
    X_exact.iloc[valid_idx]
)

- 학습 데이터로만 Target Encoding을 학습하고, 검증 데이터에는 학습된 Encoding을 적용하는 코드.

- 이를 통해 검증 데이터의 정답이 학습 과정에 섞이는 Target Leakage 방지



3. XGBoost 모델 학습 -중독 여부를 예측

In [ ]:
model=xgb.XGBClassifier(**model_params)

model.fit(
    X_fold_train,
    y[train_idx],
    eval_set=[(X_fold_valid, y[valid_idx])]
)

- XGBoost 분류 모델 생성, 학습 코드

XGBoost: 여러 개의 결정 트리를 순서대로 만들어서 이전 모델이 틀린 부분을 다음 모델이 보완하는 방식의 머신러닝 모델

- 학습 데이터로 모델 학습하면서, Validation 데이터를 이용하여 모델의 성능 확인

X_fold_train: 모델이 학습할 입력 데이터

y[train_idx]: 실제 스마트폰 중독 여부

X_fold_valid:학습 중 성능을 확인할 검증 데이터

y[valid_idx]: 검증 데이터의 실제 정답

+) "learning_rate", "max_depth", "subsample", "colsample_bytree" 등의 하이퍼파라미터 설정 / early_stoipping_rounds=200을 사용하여 검증 성능이 더 이상 좋아지지 않을 경우 학습 멈추도록 함.



# 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

1. 새롭게 알게 된 내용

- Target Encoding은 범주형 변수의 단순한 숫자 변환보다 Target과의 관계를 활용할 수 있다는 것을 알게 되었다.

(Target Encoding은 문자로 되어 있는 범주형 변수가 있을 때, 예측하려는 Target과의 관계를 이용해서 숫자로 표현하는 방법)

- Target Encoding을 잘못 적용하면 Target Leakage가 발생하여 검증 점수가 실제보다 높게 나올 수 있다는 것을 알게 되었다.

- 이를 해결하기 위해 nested Cross-Fitting을 적용하는 방법을 배웠다.

(전체 Train 데이터에 Target Encoding을 하면 학습에 사용된 Target 정보가 Encoding 값에 직접 들어가게 된다. Target Encoding 과정에서도 Cross-Fitting을 사용해서 데이터가 서로 정보를 미리 보지 않도록 만든 것이다.)

(Nested Cross-Fitting은 Outer Fold, Inner Fold 두 단계의 교차검증 구조를 사용한다. Outer Fold는 전체 데이터 학습 덷이터 / 검증 데이터로 나누는 과정을 반복하는 것이다. Inner Fold는 Outer Train 데이터 안에서 또 나눠서 Encoding을 만드는 작업이다.)

# 어려웠던 내용

- Outer Fold와 Inner Fold를 동시에 사용하는 중첨 검증 구조가 복잡했다.

- Train 데이터와 Validation 데이터에 Target Encoding을 각각 어떻게 적용해야 하는지 이해하는 데에 어려움이 있었다.

(Validation 데이터의 정답 y_valid는 Target Encoding을 만드는 데 사용하지 않는다.)

- 많은 데이터를 XGBoost로 학습하기 때문에 GPU와 메모리 관리도 중요했다.

# 배울 점

- 높은 성능보다 데이터 누수를 방지한 신뢰할 수 있는 검증이 중요하다.

(Targert Leackage가 발생하면 실제 성능보다 높은 점수가 나올 수도 있다. 검증 데이터의 정보를 학습에 사용하지 않고, 실제 새로운 데이터를 예측하는 상황과 비슷하게 평가한다.)

- 단순히 모델을 바꾸는 것뿐만 아니라, 데이터의 특성을 반영한 파생변수와 Encoding 방법을 설계하는 것이 중요하다는 점을 배웠다.

- 여러 Seed의 결과를 평균하는 Ensemble을 통해 모델의 예측을 안정화할 수 있다는 것도 알게 되었다.

(이 프로젝트에서는 SEEDS=[42, 2026]으로 두 개의 Seed를 사용했다. 두 결과를 평균한다. 즉 하나의 모델 결과에만 의존하지 않고, 서로 다른 Seed에서 학습한 모델의 예측값을 평균하면 예측을 안정화할 수 있다.)

In [ ]:
probability_oof=np.mean(
    np.vstack(seed_oof_predictions),
    axis=0
)